# Agentic Alpha High Chop Execution Validation V1

Validate close-to-close high-chop LGB/XGB signals against minute-level execution paths.

In [1]:

from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from qlib.workflow import R

ROOT_CANDIDATES = [Path.cwd(), Path('/Users/rziiiii/dev/workspace')]
ROOT = next(
    path
    for base in ROOT_CANDIDATES
    for path in [base, *base.parents]
    if (path / 'AGENTS.md').exists() and (path / 'SPEC.md').exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.intraday import fetch_futu_intraday_bars, read_intraday_cache

EXPERIMENT = 'agentic_alpha_high_chop_execution_validation_v1'
CARD = ROOT / 'research' / EXPERIMENT
SOURCE_CARD = ROOT / 'research' / 'agentic_alpha_high_chop_lgb_xgb_v1'
PROVIDER_URI = (ROOT / 'data' / 'qlib' / 'soxl').resolve()
RECORDER_URI = (CARD / 'qlib_recorder').resolve()
REGULAR_INTRADAY = ROOT / 'data' / 'processed' / 'intraday' / 'soxl_soxx_1m.parquet'
START = '2026-06-05'
SCORE_START = '2026-06-05'
END = '2026-07-02'
EXTENDED_FETCH_START = '2026-06-26'
POSITION_FLOOR = 0.01

CARD.mkdir(parents=True, exist_ok=True)
if RECORDER_URI.exists():
    shutil.rmtree(RECORDER_URI)
RECORDER_URI.mkdir(parents=True, exist_ok=True)
workflow_yaml = f'''experiment: {EXPERIMENT}
card_type: execution_validation
qlib:
  provider_uri: {PROVIDER_URI.as_posix()}
  recorder_uri: {RECORDER_URI.as_posix()}
  market: all
  region: us
source_strategy:
  card: research/agentic_alpha_high_chop_lgb_xgb_v1
  signal_artifact: .high_chop_lgb_xgb_latest_positions.csv.gz
validation:
  start: {START}
  end: {END}
  source: qlib daily + Futu regular/extended 1m
artifact_policy:
  policy: slim_card_recorder_first
  machine_entrypoint: summary.json
'''
(CARD / 'workflow.yaml').write_text(workflow_yaml, encoding='utf-8')

source_summary = json.loads((SOURCE_CARD / 'summary.json').read_text(encoding='utf-8'))
source_recorder = Path(source_summary['recorder_local_dir'])
latest_positions_path = source_recorder / 'artifacts' / 'agentic_alpha_high_chop_lgb_xgb_v1' / '.high_chop_lgb_xgb_latest_positions.csv.gz'
latest_positions = pd.read_csv(latest_positions_path)
latest_positions['datetime'] = pd.to_datetime(latest_positions['datetime']).dt.normalize()
latest_positions = latest_positions.loc[(latest_positions['datetime'] >= pd.Timestamp(SCORE_START)) & (latest_positions['datetime'] <= pd.Timestamp(END))].copy()
latest_positions = latest_positions.sort_values('datetime')
latest_positions['previous_target_position'] = latest_positions['target_position'].shift(1).fillna(0.0)
latest_positions['target_change'] = latest_positions['target_position'] - latest_positions['previous_target_position']

qlib.init(provider_uri=PROVIDER_URI.as_posix(), region='us', kernels=1, clear_mem_cache=True)
daily = D.features(
    ['SOXL'],
    ['$open', '$high', '$low', '$close', '$volume', '$vix'],
    start_time=START,
    end_time=END,
    freq='day',
).reset_index().sort_values('datetime')
daily['datetime'] = pd.to_datetime(daily['datetime']).dt.normalize()
daily['next_datetime'] = daily['datetime'].shift(-1)
daily['next_close'] = daily['$close'].shift(-1)
daily['next_open'] = daily['$open'].shift(-1)
daily['close_to_close_return'] = daily['next_close'] / daily['$close'] - 1.0
daily['close_to_next_open_gap'] = daily['next_open'] / daily['$close'] - 1.0

regular = read_intraday_cache(REGULAR_INTRADAY)
regular = regular.loc[(regular['symbol'] == 'SOXL') & (regular['session'] == 'regular')].copy()
regular['datetime'] = pd.to_datetime(regular['date']).dt.normalize()
regular_summary = []
for day, bars in regular.loc[(regular['datetime'] >= pd.Timestamp(START)) & (regular['datetime'] <= pd.Timestamp(END))].groupby('datetime'):
    bars = bars.sort_values('timestamp')
    regular_summary.append(
        {
            'datetime': day,
            'rth_open_1m': float(bars['open'].iloc[0]),
            'rth_close_1m': float(bars['close'].iloc[-1]),
            'rth_low_1m': float(bars['low'].min()),
            'rth_high_1m': float(bars['high'].max()),
            'rth_volume_1m': float(bars['volume'].sum()),
            'rth_return_open_to_close': float(bars['close'].iloc[-1] / bars['open'].iloc[0] - 1.0),
            'rth_low_from_open': float(bars['low'].min() / bars['open'].iloc[0] - 1.0),
            'rth_low_from_close': float(bars['low'].min() / bars['close'].iloc[-1] - 1.0),
        }
    )
regular_summary = pd.DataFrame(regular_summary)

extended = fetch_futu_intraday_bars('SOXL', start=EXTENDED_FETCH_START, end=END, extended_time=True, session='ALL')
extended_artifact = CARD / '.intraday_extended_soxl_recent.csv.gz'
extended.to_csv(extended_artifact, index=False, compression='gzip')
extended['datetime'] = pd.to_datetime(extended['date']).dt.normalize()
after = extended.loc[extended['session'] == 'afterhours'].copy()
after_summary = []
for day, bars in after.groupby('datetime'):
    bars = bars.sort_values('timestamp')
    rth_close_rows = regular_summary.loc[regular_summary['datetime'] == day, 'rth_close_1m']
    rth_close = float(rth_close_rows.iloc[0]) if not rth_close_rows.empty else np.nan
    after_summary.append(
        {
            'datetime': day,
            'afterhours_first_open': float(bars['open'].iloc[0]),
            'afterhours_close': float(bars['close'].iloc[-1]),
            'afterhours_low': float(bars['low'].min()),
            'afterhours_high': float(bars['high'].max()),
            'afterhours_volume': float(bars['volume'].sum()),
            'afterhours_first_gap_from_rth_close': float(bars['open'].iloc[0] / rth_close - 1.0) if rth_close else np.nan,
            'afterhours_close_return_from_rth_close': float(bars['close'].iloc[-1] / rth_close - 1.0) if rth_close else np.nan,
            'afterhours_mae_from_rth_close': float(bars['low'].min() / rth_close - 1.0) if rth_close else np.nan,
            'afterhours_mfe_from_rth_close': float(bars['high'].max() / rth_close - 1.0) if rth_close else np.nan,
        }
    )
after_summary = pd.DataFrame(after_summary)

next_open = regular_summary[['datetime', 'rth_open_1m']].rename(columns={'datetime': 'next_datetime', 'rth_open_1m': 'next_rth_open_1m'})
validation = (
    latest_positions.merge(daily, on='datetime', how='left')
    .merge(regular_summary, on='datetime', how='left')
    .merge(after_summary, on='datetime', how='left')
    .merge(next_open, on='next_datetime', how='left')
)
validation['positioned_close_to_close_return'] = validation['target_position'] * validation['close_to_close_return']
validation['positioned_close_to_next_open_gap'] = validation['target_position'] * (validation['next_rth_open_1m'] / validation['rth_close_1m'] - 1.0)
validation['turnover_afterhours_first_slippage'] = validation['target_change'].abs() * validation['afterhours_first_gap_from_rth_close'].abs()
validation['positioned_afterhours_mae'] = validation['target_position'] * validation['afterhours_mae_from_rth_close']
validation['execution_status'] = np.select(
    [
        validation['target_position'].abs() < POSITION_FLOOR,
        validation['afterhours_first_open'].notna(),
        validation['next_rth_open_1m'].notna(),
    ],
    ['near_cash', 'afterhours_path_available', 'next_open_path_available'],
    default='needs_data',
)
validation_columns = [
    'datetime',
    'target_position',
    'previous_target_position',
    'target_change',
    'probability',
    'raw_target',
    '$close',
    'next_datetime',
    'next_close',
    'close_to_close_return',
    'positioned_close_to_close_return',
    'rth_open_1m',
    'rth_close_1m',
    'rth_low_1m',
    'rth_high_1m',
    'rth_return_open_to_close',
    'rth_low_from_open',
    'afterhours_first_open',
    'afterhours_close',
    'afterhours_low',
    'afterhours_high',
    'afterhours_volume',
    'afterhours_first_gap_from_rth_close',
    'afterhours_close_return_from_rth_close',
    'afterhours_mae_from_rth_close',
    'afterhours_mfe_from_rth_close',
    'next_rth_open_1m',
    'positioned_close_to_next_open_gap',
    'turnover_afterhours_first_slippage',
    'positioned_afterhours_mae',
    'execution_status',
]
validation = validation.loc[:, validation_columns]
validation['datetime'] = validation['datetime'].dt.date.astype(str)
validation['next_datetime'] = pd.to_datetime(validation['next_datetime']).dt.date.astype('string')
validation.to_csv(CARD / 'execution_validation.csv', index=False)

recent = validation.loc[validation['datetime'] >= '2026-06-26'].copy()
latest_row = validation.loc[validation['datetime'] == END].iloc[0].to_dict()
summary_payload = {
    'workflow': 'agentic_alpha_high_chop_execution_validation',
    'source_strategy_card': 'research/agentic_alpha_high_chop_lgb_xgb_v1',
    'source_strategy_recorder_id': source_summary['recorder_id'],
    'source_signal_artifact': latest_positions_path.relative_to(ROOT).as_posix(),
    'provider_uri': PROVIDER_URI.as_posix(),
    'regular_intraday_source': REGULAR_INTRADAY.relative_to(ROOT).as_posix(),
    'extended_intraday_source': 'futu.OpenQuoteContext.request_history_kline(K_1M, extended_time=True, session=ALL)',
    'as_of': END,
    'coverage_start': SCORE_START,
    'coverage_end': END,
    'rows': int(len(validation)),
    'recent_rows': int(len(recent)),
    'latest': {
        'date': latest_row['datetime'],
        'target_position': float(latest_row['target_position']),
        'probability': float(latest_row['probability']),
        'execution_status': latest_row['execution_status'],
        'rth_return_open_to_close': float(latest_row['rth_return_open_to_close']),
        'rth_low_from_open': float(latest_row['rth_low_from_open']),
        'afterhours_first_gap_from_rth_close': float(latest_row['afterhours_first_gap_from_rth_close']),
        'afterhours_mae_from_rth_close': float(latest_row['afterhours_mae_from_rth_close']),
        'afterhours_close_return_from_rth_close': float(latest_row['afterhours_close_return_from_rth_close']),
    },
    'risk_checks': {
        'near_cash_days_recent': int((recent['target_position'].abs() < POSITION_FLOOR).sum()),
        'max_abs_target_recent': float(recent['target_position'].abs().max()),
        'max_afterhours_mae_recent': float(recent['afterhours_mae_from_rth_close'].min()),
        'max_rth_low_from_open_recent': float(recent['rth_low_from_open'].min()),
        'needs_next_open_data_rows': int(recent['next_rth_open_1m'].isna().sum()),
    },
    'decision': 'execution_not_proven',
    'decision_reason': 'close-to-close daily signal needs before-cutoff generation plus afterhours/next-open fill simulation; latest 2026-07-02 target is near cash, but this does not prove live execution ability.',
}
(CARD / 'execution_validation_summary.json').write_text(json.dumps(summary_payload, indent=2, sort_keys=True), encoding='utf-8')
(CARD / 'failure_modes.md').write_text(
    '# Failure Modes\n\n'
    '- close_to_close_execution_gap: target_position(t) is evaluated on t to t+1 close-to-close return; live use requires a timestamped signal before the chosen close/after-hours/next-open execution cutoff.\n'
    '- recent_near_cash_not_proof: 2026-06-26 through 2026-07-02 is near cash, so it avoids the July 2 drawdown but does not prove fills for high-turnover full-cash/full-long flips.\n'
    '- minute_history_depth: local Futu 1m coverage starts in 2024, so execution validation cannot close the 2019-2026 promotion gate without a longer intraday vendor.\n',
    encoding='utf-8',
)

R.set_uri(RECORDER_URI.as_posix())
with R.start(experiment_name=EXPERIMENT):
    recorder = R.get_recorder()
    recorder_id = recorder.id
    R.log_params(
        source_strategy='agentic_alpha_high_chop_lgb_xgb_v1',
        source_recorder_id=source_summary['recorder_id'],
        provider_uri=PROVIDER_URI.as_posix(),
        validation_start=SCORE_START,
        validation_end=END,
    )
    for key, value in summary_payload['risk_checks'].items():
        if isinstance(value, (int, float)):
            R.log_metrics(**{key: value})
    for artifact in [
        CARD / 'execution_validation.csv',
        CARD / 'execution_validation_summary.json',
        CARD / 'failure_modes.md',
        CARD / 'workflow.yaml',
        extended_artifact,
    ]:
        recorder.log_artifact(artifact.as_posix(), artifact_path=EXPERIMENT)
    recorder_local_dir = Path(recorder.get_local_dir())
for transient_name in ['code_cached.txt', 'code_diff.txt', 'code_status.txt']:
    transient_path = recorder_local_dir / 'artifacts' / transient_name
    if transient_path.exists():
        transient_path.unlink()
extended_artifact.unlink(missing_ok=True)

recorder_artifacts = sorted(path.relative_to(recorder_local_dir).as_posix() for path in recorder_local_dir.rglob('*') if path.is_file())
summary_payload['recorder_id'] = recorder_id
summary_payload['recorder_local_dir'] = recorder_local_dir.as_posix()
summary_payload['recorder_artifacts'] = recorder_artifacts
(CARD / 'execution_validation_summary.json').write_text(json.dumps(summary_payload, indent=2, sort_keys=True), encoding='utf-8')
summary = {
    'experiment': EXPERIMENT,
    'card_type': 'execution_validation',
    'workflow_config': 'workflow.yaml',
    'provider_uri': PROVIDER_URI.as_posix(),
    'recorder_uri': RECORDER_URI.as_posix(),
    'recorder_id': recorder_id,
    'recorder_local_dir': recorder_local_dir.as_posix(),
    'recorder_artifacts': recorder_artifacts,
    'market': 'all',
    'region': 'us',
    'as_of': END,
    'source_strategy': {
        'card': 'research/agentic_alpha_high_chop_lgb_xgb_v1',
        'recorder_id': source_summary['recorder_id'],
        'latest_signal': source_summary['latest_signal'],
    },
    'qlib_native_api': ['qlib.init', 'qlib.data.D.features', 'qlib.workflow.R.start'],
    'external_data_api': ['futu.OpenQuoteContext.request_history_kline(K_1M, extended_time=True, session=ALL)'],
    'qlib_docs': {
        'data': 'https://qlib.readthedocs.io/en/latest/component/data.html',
        'workflow': 'https://qlib.readthedocs.io/en/latest/component/workflow.html',
    },
    'validation_summary': summary_payload,
    'promotion_decision': 'execution_not_proven',
    'sidecars': {
        'execution_validation': 'execution_validation.csv',
        'execution_validation_summary': 'execution_validation_summary.json',
        'failure_modes': 'failure_modes.md',
    },
}
summary['artifact_policy'] = {
    'policy': 'slim_card_recorder_first',
    'primary_artifact': f'{EXPERIMENT}.ipynb',
    'machine_entrypoint': 'summary.json',
    'workflow_config': 'workflow.yaml',
    'kept_sidecars': sorted(summary['sidecars'].values()),
    'reproducible_from_recorder': True,
    'recorder_only_artifacts': {
        'extended_intraday_recent': f'artifacts/{EXPERIMENT}/.intraday_extended_soxl_recent.csv.gz',
    },
}
(CARD / 'summary.json').write_text(json.dumps(summary, indent=2, sort_keys=True), encoding='utf-8')
conclusion = f'''# 中文结论

- 做了什么：读取 high-chop LGB/XGB 模型的最新 target_position、qlib 日线、Futu regular/extended 1m，检查 close-to-close 信号在收盘、盘后和次日开盘路径上的执行风险。
- 7/2 结论：2026-07-02 target_position={summary_payload['latest']['target_position']:.4f}，属于 near_cash；当天 RTH open-to-close={summary_payload['latest']['rth_return_open_to_close']:.2%}、RTH low-from-open={summary_payload['latest']['rth_low_from_open']:.2%}、盘后首笔相对 RTH close={summary_payload['latest']['afterhours_first_gap_from_rth_close']:.2%}、盘后 MAE={summary_payload['latest']['afterhours_mae_from_rth_close']:.2%}。这说明修复日线后模型没有在 7/2 满仓承受暴跌。
- 近期窗口：2026-06-26 到 2026-07-02 全部 {summary_payload['risk_checks']['near_cash_days_recent']} 个交易日 near_cash，max_abs_target={summary_payload['risk_checks']['max_abs_target_recent']:.4f}；这能解释为什么 7/2 没吃大回撤，但不能证明满仓/空仓切换的实盘能力。
- 是否晋升：execution_not_proven；close-to-close 回测仍需要在固定信号生成时间、盘后/次日开盘成交假设、滑点和分钟 MAE/MFE 上继续验证。
- qlib recorder：{recorder_id}
'''
(CARD / 'conclusion.md').write_text(conclusion, encoding='utf-8')
print(conclusion)


[87215:MainThread](2026-07-03 11:41:05,507) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[87215:MainThread](2026-07-03 11:41:05,508) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[87215:MainThread](2026-07-03 11:41:05,509) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


2026-07-03 11:41:05,759 | 87215 | 8676912192 | [open_context_base.py:409] _init_connect_sync: New connect ready: conn=7478654023012798224(1) context=<futu.quote.open_quote_context.OpenQuoteContext object at 0x11789e1d0>


2026-07-03 11:41:06,980 | 87215 | 6325284864 | [open_context_base.py:516] on_disconnect: Disconnected: conn=0(1) reason=CallClose


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[87215:MainThread](2026-07-03 11:41:07,103) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name agentic_alpha_high_chop_execution_validation_v1.


[87215:MainThread](2026-07-03 11:41:07,104) INFO - qlib.workflow - [exp.py:258] - Experiment 917265902071001972 starts running ...


[87215:MainThread](2026-07-03 11:41:07,160) INFO - qlib.workflow - [recorder.py:345] - Recorder dad036329bc94f82b0fee93d658f7bb1 starts running under Experiment 917265902071001972 ...


[87215:MainThread](2026-07-03 11:41:07,484) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | waiting `async_log` Done


# 中文结论

- 做了什么：读取 high-chop LGB/XGB 模型的最新 target_position、qlib 日线、Futu regular/extended 1m，检查 close-to-close 信号在收盘、盘后和次日开盘路径上的执行风险。
- 7/2 结论：2026-07-02 target_position=0.0005，属于 near_cash；当天 RTH open-to-close=-17.22%、RTH low-from-open=-22.97%、盘后首笔相对 RTH close=-0.24%、盘后 MAE=-0.81%。这说明修复日线后模型没有在 7/2 满仓承受暴跌。
- 近期窗口：2026-06-26 到 2026-07-02 全部 5 个交易日 near_cash，max_abs_target=0.0005；这能解释为什么 7/2 没吃大回撤，但不能证明满仓/空仓切换的实盘能力。
- 是否晋升：execution_not_proven；close-to-close 回测仍需要在固定信号生成时间、盘后/次日开盘成交假设、滑点和分钟 MAE/MFE 上继续验证。
- qlib recorder：dad036329bc94f82b0fee93d658f7bb1

